In [8]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [9]:
# experiment_file = '../../experiments/parallelized_experiments/output/mirror_cosine_dash/2024_01_08_11_21//experiment_result.json'

# stiffness_path = '../../experiments/parallelized_experiments/output/mirror_cosine_dash/2024_01_08_11_21'
# name = 'mirror_cosine_dash'

In [10]:
experiment_file = '../../experiments/parallelized_experiments/output/dash_line/2024_01_18_13_03//experiment_result.json'

stiffness_path = '../../experiments/parallelized_experiments/output/dash_line/2024_01_18_13_03/'
name = 'dash_line'

### Overview

In [11]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [12]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [13]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [14]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [15]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [16]:
invalid_tags = np.array(df['name'][df['Planar equilibrium'] != 1])

In [17]:
kappa_path = None

In [20]:
radius = np.array(data['pattern_parameters'][0]['values'])
angles = np.array(data['pattern_parameters'][1]['values'])

In [23]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [24]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [25]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [26]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [27]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [28]:
np.argmax(min_bending_stiffness)

In [29]:
valid_tags[44]

In [30]:
11 * 19

In [31]:
len(angles), len(radius)

In [35]:
samples = np.array([[float(n) for n in tag.split('_')[:]] for tag in valid_tags])

In [36]:
plt.scatter(samples[:, 0], samples[:, 1])

In [37]:
def show_tags(threshold = 0):
    curr_tags = valid_tags[(np.where(min_bending_stiffness > threshold))]
    samples = np.array([[float(n) for n in tag.split('_')[:]] for tag in curr_tags])
    plt.scatter(samples[:, 0], samples[:, 1])

In [38]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [39]:
radius

In [41]:
interact(show_tags, threshold=widgets.FloatSlider(min=0, max=2.5, step=0.1, value=0));

### Get scale function convex hull

In [43]:
import matplotlib.cm as cm
import matplotlib as mpl

In [44]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [45]:
hull

In [46]:
import matplotlib.pyplot as plt
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
plt.plot(points[hull.vertices,0], points[hull.vertices,1], 'r--', lw=2)
plt.plot(points[hull.vertices[0],0], points[hull.vertices[0],1], 'ro')
plt.show()

In [47]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

# plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)


points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

ax.title.set_text("Scale factors")
plt.xlabel("x scale factors")
plt.ylabel("y scale factors")

plt.scatter(max_scale_factors, min_scale_factors, label = 'min_stiffness', s = 200, alpha = 1, c = min_bending_stiffness)
# plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

### Validate the max and min scale factors are aligned with the x and y axis

In [48]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [49]:
eqns = hull.equations

In [50]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [51]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [37]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [38]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [39]:
plt.plot(stiffness_coefficients[:, 4])

In [40]:
np.set_printoptions(suppress=True, precision=4)

In [41]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [42]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [43]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [53]:
stiffness_coefficients

In [44]:
grid_data = np.zeros((9, len(radius), len(angles)))

for i in range(len(valid_tags)):
    radius_index = int(i / len(angles))
    angles_index = int(i % len(angles))

    grid_data[0][radius_index][angles_index] = max_scale_factors[i]
    grid_data[1][radius_index][angles_index] = min_scale_factors[i]
    grid_data[2][radius_index][angles_index] = x_scale_factors[i]
    grid_data[3][radius_index][angles_index] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][radius_index][angles_index] = stiffness_coefficients[i][s]

In [45]:
importlib.reload(parametrization_helper)

In [46]:
len(grid_data.shape) - 1

In [47]:
grid_data.shape

In [48]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, radius, angles)

In [49]:
scale_factors_grid_data = np.zeros((2, len(radius), len(angles)))
for i in range(len(valid_tags)):
    radius_index = int(i / len(angles))
    angles_index = int(i % len(angles))
    scale_factors_grid_data[0][radius_index][angles_index] = x_scale_factors[i]
    scale_factors_grid_data[1][radius_index][angles_index] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, radius, angles)

In [50]:
radius, angles

In [52]:
grid_data[4]

In [51]:
import numpy as np

# Generate 2D test parameters
x = np.linspace(0.5, 2.3, 100)
y = np.linspace(45, 75, 100)
test_parameters_x, test_parameters_y = np.meshgrid(x, y)

titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
fig, axes = plt.subplots(1, 7, figsize=(45, 8))
index = [0, 1, 2, 3, 4, 7, 8]

for i in range(len(index)):
    
    # Evaluate the spline at the 2D test parameters
    z = splines[index[i] * 3 + 0]([test_parameters_x.flatten(), test_parameters_y.flatten()])
    z = z.reshape((100i, 100))

    # Use imshow to visualize the 2D data
    im = axes[i].imshow(z, extent=[0.5, 2.3, 45, 75], origin='lower', aspect='auto', cmap='coolwarm')
    axes[i].set_title(titles[index[i]], fontsize=21)

    # Add a colorbar to each subplot
    fig.colorbar(im, ax=axes[i])

In [ ]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [ ]:
stiffness_coefficients.shape

### End data generating

### Parametrization

In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [ ]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
from py_newton_optimizer import NewtonOptimizerOptions

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [ ]:
import utils, mesh_utilities
importlib.reload(utils)

In [ ]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
lines = np.array(eqns)

### New local global with convex hull

In [ ]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)
lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [ ]:
default_pattern_params = np.array([1.0]  * len(lg.getAlphas()) + [50]  * len(lg.getAlphas()))

In [ ]:
# mat_info = np.array(default_pattern_params).reshape((2, len(lg.getAlphas())))

In [ ]:
default_pattern_params

In [ ]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.6, 2.2], [45, 75]])
rparam.patternParamNormalizationFactors = np.array([1.6, 30])
rparam.diffRegW = 0.0

In [ ]:
rparam.patternRegP

In [ ]:
visualization.visualize_both(rparam, height = 4)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [ ]:
rparam.bendRegW = 1
rparam.patternRegW = patternRegW

In [ ]:
rparam.energy(PET.RGP)

In [ ]:
rparam.energy(PET.Bending)

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
def optimize_rparam(param, patternRegW, phiRegW, bendRegW = 0.0, update_uv = True, niter = 100):
    param.patternRegW = patternRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = niter
    opts.gradTol = 1e-9
    opts.factorizer = opts.factorizer.CatamariNesdis
    benchmark.reset()
    
    if update_uv:
        fixedvars = [param.uOffset(), param.vOffset(), param.phiOffset()]
    else:
        fixedvars = range(param.stretchOffset())

    cr = parametrization.pattern_parametrization_knitro(param, opts.niter, fixedvars)
    benchmark.report()
    return cr

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, 0, 0, bendRegW = 0, update_uv = False, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
parametrization_helper.visualize_scale_factors(eqns, rparam.getAlphas(), rparam.getBetas())

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=2)

In [ ]:
rparam.phiRegW = 5e-6
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 0, phiRegW = 5e-6, bendRegW = 0, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=2, width = 30)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
rparam.patternRegW = 1e-6
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-6, phiRegW = 5e-6, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-6, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-6, bendRegW = 0, update_uv = True, niter = 100)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=2, width = 30)

### Bending

In [ ]:
rparam.bendRegW = 2e-5
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-6, bendRegW = 2e-5, update_uv = False, niter = 1000)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=2, width = 30)

In [ ]:
benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-6, bendRegW = 2e-5, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-7, bendRegW = 5e-6, update_uv = True, niter = 200)
benchmark.report()

benchmark.reset()
with suppress_stdout(): report = optimize_rparam(rparam, patternRegW = 1e-7, phiRegW = 1e-7, bendRegW = 1e-6, update_uv = True, niter = 200)
benchmark.report()

In [ ]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType
names = ["Full", "PReg", "Bend", "Fitt", "SReg", "PhiR"]
energy_values = list(map(rparam.energy, [PET.Full, PET.PatternRegularization, PET.Bending, PET.DEBUG_Fitting, PET.DEBUG_StretchRegularization, PET.DEBUG_PhiRegularization]))
list(zip(names, energy_values))

In [ ]:
importlib.reload(visualization)
visualization.visualize_both(rparam, height = 4)
visualization.visualize_pattern(rparam, height = 4, num_pattern_vars=2)

In [ ]:
importlib.reload(parametrization_helper)
parametrization_helper.visualize_scale_factors(lines, rparam.getAlphas(), rparam.getBetas())

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False, width = 5, height = 5)

In [ ]:
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_TRI, orientationHue=False, width = 10, height = 10)

## Upsampling and channel generation

In [ ]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [ ]:
def fusing_curve_polyline(patternParams):
#     Draw dash_line
    radius = patternParams[0]
    angle = patternParams[1]
    
    dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * radius + np.array([0, 0])
    mid_point = np.array([0, 0])
    dash_line = (np.array([dash_point, mid_point * 2 - dash_point]) + [2.5, 2.5]) / 5 * np.pi
    return dash_line

In [ ]:
boundary_vertices = [[0, 0], [np.pi, 0], [np.pi, np.pi], [0, np.pi]]

In [ ]:
boundary_edges = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + 2

In [ ]:
visualization.plot_line_segments(list(fusing_curve_polyline([0.5, 50])) + boundary_vertices, [[0, 1]] + list(boundary_edges))

In [ ]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines, boundaryVxs, boundaryEdges = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 7, frequency=0.15, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0])

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 10, height=10)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 10, height = 10)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(boundaryEdges + len(sheet_vxs)), width = 5, height = 5)
plt.scatter(boundaryVxs[boundaryEdges[:, 0], 0], boundaryVxs[boundaryEdges[:, 0], 1], c = np.arange(len(boundaryVxs)), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [ ]:
import mesher_helper
importlib.reload(mesher_helper)

In [ ]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [ ]:
np.save("boundary.npy", boundaryVxs[boundaryEdges[:, 0]])

In [ ]:
np.save("sheet_vxs.npy", sheet_vxs)
np.save("concatenated_polylines.npy", concatenated_polylines)

In [ ]:
boundary = boundaryVxs[boundaryEdges[:, 0]][:, :2].tolist()
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_svg(boundary, polylines, 'sheet_pattern_{}.svg'.format(time_stamp))

In [ ]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[boundaryEdges[:, 0]], sheet_vxs, concatenated_polylines, gui = False)

In [ ]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [ ]:
fusing_data, new_fusing

In [ ]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

In [ ]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [ ]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [ ]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [ ]:
isheet.pressure = 1e-5

In [ ]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [ ]:
isheet.pressure = 5e-2

In [ ]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Generate Fabrication Files

In [ ]:
old_to_new = np.arange(np.max(isheet.wallVertices()) + 1)

In [ ]:
old_to_new[isheet.wallVertices()] = np.arange(len(isheet.wallVertices()))

In [ ]:
from parametrization_helper import form_polylines

In [ ]:
result_vxs = isheet.restWallVertexPositions()
result_edges = old_to_new[isheet.wallBoundaryEdges()]
result_edges = form_polylines(result_edges.tolist())
concatenated_polylines = []
for polyline in result_edges:
    concatenated_polylines.extend(polyline)


In [ ]:
visualization.plot_line_segments(result_vxs, concatenated_polylines)